# 📄 RAG Document Chatbot
Answers questions and summarizes any PDF using Retrieval-Augmented Generation.

Pipeline: `PDF → Chunk → Embed → FAISS → Query → Qwen 3.6 27B via Groq → Answer`

In [1]:
!pip install pdfplumber

import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 39.9 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


## 1. PDF Extraction
Extract text from PDF page by page using `pdfplumber`.

In [2]:
# Upload a PDF first
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
document_text = extract_text_from_pdf(pdf_path)

print(document_text[:500])  # preview first 500 characters
print(f"\nTotal characters: {len(document_text)}")

Saving Artificial Intelligence in Healthcare.pdf to Artificial Intelligence in Healthcare.pdf
Artificial Intelligence in Healthcare
Introduction
Artificial Intelligence (AI) has become one of the most important technologies in modern
healthcare. AI systems can analyze vast amounts of medical data, assist healthcare
professionals in diagnosis, recommend treatment plans, and improve hospital management.
The combination of machine learning, deep learning, natural language processing, and
computer vision enables healthcare organizations to provide faster, more accurate, and
personalized pati

Total characters: 6415


## 2. Text Chunking
Split text into 500-char overlapping chunks. Smaller chunks = more precise retrieval. Overlap prevents context loss at boundaries.

In [3]:
!pip install langchain-text-splitters -q

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(document_text)

print(f"Total chunks created: {len(chunks)}")
print("\nFirst chunk:")
print(chunks[0])

Total chunks created: 15

First chunk:
Artificial Intelligence in Healthcare
Introduction
Artificial Intelligence (AI) has become one of the most important technologies in modern
healthcare. AI systems can analyze vast amounts of medical data, assist healthcare
professionals in diagnosis, recommend treatment plans, and improve hospital management.
The combination of machine learning, deep learning, natural language processing, and
computer vision enables healthcare organizations to provide faster, more accurate, and


## 3. Embeddings
Convert each chunk into a 384-dim vector using `all-MiniLM-L6-v2` — runs locally, no API needed. Similar meanings → similar vectors.

In [4]:
!pip install Pillow==10.4.0 sentence-transformers faiss-cpu -q

from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model (runs locally, free, no API needed)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Convert all chunks into embeddings
chunk_embeddings = embedder.encode(chunks)

print(f"Shape of embeddings: {chunk_embeddings.shape}")
print(f"Each chunk became a vector of {chunk_embeddings.shape[1]} numbers")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 10.4.0 which is incompatible.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape of embeddings: (15, 384)
Each chunk became a vector of 384 numbers


## 4. FAISS Vector Store
Index all embeddings for fast similarity search. Given a query vector, FAISS finds the closest matching chunks in milliseconds.

In [5]:
import faiss

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

print(f"Total vectors stored in FAISS: {index.ntotal}")

Total vectors stored in FAISS: 15


## 5. Semantic Search
Embed the query the same way, then retrieve top-k most similar chunks from FAISS.

In [6]:
def search(query, top_k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)

    results = [chunks[i] for i in indices[0]]
    return results

query = "What is this project about?"
results = search(query)

for i, r in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(r)
    print()

--- Result 1 ---
outcomes and reduce operational costs.
The future of healthcare will likely involve close collaboration between healthcare
professionals and intelligent AI systems, resulting in more efficient, accurate, and personalized
patient care.

--- Result 2 ---
Personalized treatments
Faster medical research
These advantages contribute to beter healthcare services worldwide.
Future Scope
Future healthcare AI systems are expected to integrate with wearable devices, smart
hospitals, and telemedicine platforms.Researchers are working on explainable AI models that
provide understandable reasoning behind medical decisions.
Generative AI assistants may help doctors prepare clinical documentation, summarize patient
histories, and answer medical questions.

--- Result 3 ---
Artificial Intelligence in Healthcare
Introduction
Artificial Intelligence (AI) has become one of the most important technologies in modern
healthcare. AI systems can analyze vast amounts of medical data, assist hea

## 6. RAG Q&A
Retrieve relevant chunks → insert as context → send to Qwen 3.6 27B via Groq. Model is instructed to say "I don't know" rather than hallucinate.


In [7]:
!pip install groq -q

import re
from groq import Groq

client = Groq(api_key="YOUR_GROQ_API_KEY")

def clean_answer(text: str) -> str:
    if not text:
        return text
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()

def ask_question(query, top_k=3):
    relevant_chunks = search(query, top_k)
    context = "\n\n".join(relevant_chunks)

    prompt = f"""Answer the question based only on the context below.
If the answer isn't in the context, say "I don't have that information in the document."

Context:
{context}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[{"role": "user", "content": prompt}],
        reasoning_format="hidden"
    )

    return clean_answer(response.choices[0].message.content)

answer = ask_question("What is Artificial Intelligence in healthcare?")
print(answer)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.5 MB/s eta 0:00:00
Based on the provided context, Artificial Intelligence in healthcare is one of the most important modern technologies that uses AI systems to analyze vast amounts of medical data, assist healthcare professionals in diagnosis, recommend treatment plans, and improve hospital management. It combines machine learning, deep learning, natural language processing, and computer vision to enable faster, more accurate, and personalized patient care while reducing operational costs and transforming areas such as medical imaging and drug discovery.


## 7. Summarization
Full document summary uses entire text. Section summary retrieves top-5 chunks for a given topic then summarizes those.

In [8]:
def summarize_document():
    """Summarize the entire document"""
    # Use first few chunks as sample if document is huge, or all if small
    full_text = " ".join(chunks)  # or document_text if you want the whole thing

    prompt = f"""Summarize the following document in a clear, structured way.
Include:
1. Main topic/purpose
2. Key sections covered
3. Important conclusions or findings

Document:
{full_text[:8000]}

Summary:"""

    response = client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[{"role": "user", "content": prompt}],
        reasoning_format="hidden"
    )
    return clean_answer(response.choices[0].message.content)


def summarize_section(section_query, top_k=5):
    """Summarize a specific section by finding relevant chunks first"""
    relevant_chunks = search(section_query, top_k)
    context = "\n\n".join(relevant_chunks)

    prompt = f"""Summarize the following section of a document clearly and concisely.

Section content:
{context}

Summary:"""

    response = client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[{"role": "user", "content": prompt}],
        reasoning_format="hidden"
    )
    return clean_answer(response.choices[0].message.content)


# Test full document summary
print("=== FULL DOCUMENT SUMMARY ===")
print(summarize_document())

print("\n\n=== SECTION SUMMARY (Load Balancing) ===")
print(summarize_section("load balancing"))

=== FULL DOCUMENT SUMMARY ===
**1. Main Topic/Purpose**  
The document examines the transformative impact of Artificial Intelligence (AI) on modern healthcare. Its purpose is to outline how AI technologies are reshaping medical diagnostics, treatment planning, hospital operations, and drug development, while also addressing the technical, ethical, and regulatory challenges involved in their adoption.

**2. Key Sections Covered**  
- **Introduction & Evolution:** Traces AI's development from 1970s rule-based expert systems to modern machine learning and deep learning models that continuously improve from medical data.  
- **Core Applications:**  
  - *Machine Learning:* Disease prediction, patient risk assessment, and hospital resource optimization.  
  - *Medical Imaging:* AI-assisted analysis of X-rays, MRIs, CT scans, and ultrasounds to detect abnormalities quickly and accurately.  
  - *Electronic Health Records (EHRs) & NLP:* Extraction and summarization of unstructured clinical no

## 8. Interactive Chat
Ask multiple questions in sequence. Type `exit` to stop.

In [9]:
print("📄 RAG Document Chatbot — Ask questions about your PDF")
print("Type 'exit' to stop\n")

while True:
    query = input("You: ")
    if query.lower() == "exit":
        print("Goodbye!")
        break

    answer = ask_question(query)
    print(f"Bot: {answer}\n")

📄 RAG Document Chatbot — Ask questions about your PDF
Type 'exit' to stop

You: What are the applications of machine learning in healthcare?
Bot: Based on the context, the applications of machine learning in healthcare include:
- Disease prediction
- Medical image classification
- Drug discovery
- Patient risk assessment
- Hospital resource optimization

You: Explain AI in medical imaging
Bot: Based on the provided context, AI is used to efficiently process and analyze medical images, identify patterns that may not be immediately visible to medical professionals, detect diseases, and predict patient outcomes with remarkable accuracy. Additionally, medical image classification is specifically listed as a key machine learning application in healthcare.

You: What information is stored in Electronic Health Records?
Bot: Based on the provided context, Electronic Health Records contain valuable patient information including:
- Medical history
- Allergies
- Prescriptions
- Diseases

You:  Wh

## Key Learnings
- Embeddings enable semantic search — exact wording doesn't need to match
- Retrieval quality determines answer quality
- Explicit grounding prevents hallucination
- Local embeddings = no API cost or rate limits